In [0]:
# Cell 1
%pip install prophet

In [0]:
# Cell 2
import os, sys, uuid
sys.path.insert(0, os.path.abspath(".."))

from ecom.audit import Audit
from ecom.forecast_train import run_forecast_training

dbutils.widgets.text("catalog", "ecom_dev")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("bundle_target", "dev")
dbutils.widgets.dropdown("promote", "false", ["false", "true"])

catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())
bundle_target = dbutils.widgets.get("bundle_target")
promote = dbutils.widgets.get("promote") == "true"
triggered_by = spark.sql("SELECT current_user()").first()[0]

In [0]:
# Cell 3
audit = Audit(spark, catalog, run_id, "forecast_train", bundle_target, triggered_by)
run_forecast_training(spark, catalog, run_id, audit, promote)

In [0]:
%sql
SELECT grain, forecast_date, horizon_days, round(yhat) AS forecast, round(yhat_lower) AS low, round(yhat_upper) AS high
FROM ecom_dev.ml.sales_forecast
WHERE grain = 'month'
  AND generated_at = (SELECT max(generated_at) FROM ecom_dev.ml.sales_forecast)
ORDER BY forecast_date;

In [0]:
%sql
-- Which forecast runs exist, and which model version made each?
SELECT forecast_run_id, model_name, model_version, generated_at, count(*) AS rows
FROM ecom_dev.ml.sales_forecast
GROUP BY forecast_run_id, model_name, model_version, generated_at
ORDER BY generated_at;

In [0]:
%sql
-- What did the training and scoring runs report?
SELECT run_id, table_name, status, rows_read, rows_written, target_version, error_message
FROM ecom_dev.control.pipeline_run_audit
WHERE layer = 'ml';

In [0]:
# Where does the 'champion' alias point?
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
name = "ecom_dev.ml.sales_forecast_prophet"
client = MlflowClient()
print("champion is currently version", client.get_model_version_by_alias(name, "champion").version)

In [0]:
client.set_registered_model_alias(name, "champion", 2)
print("champion is now version", client.get_model_version_by_alias(name, "champion").version)

In [0]:
%sql
SELECT grain, forecast_date, horizon_days, round(yhat) AS forecast, round(yhat_lower) AS low, round(yhat_upper) AS high
FROM ecom_dev.ml.sales_forecast
WHERE grain = 'month'
  AND generated_at = (SELECT max(generated_at) FROM ecom_dev.ml.sales_forecast)
ORDER BY forecast_date;

In [0]:
%sql
SELECT model_version, generated_at, grain, count(*) AS rows
FROM ecom_dev.reporting.forecast_latest
GROUP BY model_version, generated_at, grain ORDER BY grain;